# HelioBus800 — Fine-tuning de Agentes no Google Colab

Roda aqui, baixa o modelo e incorpora ao repo.

In [ ]:
# 1. Instalar dependências
!pip install -q unsloth transformers datasets peft trl huggingface_hub


In [ ]:
# 2. Dataset de treino dos agentes HB800 (zero alucinação)
training_data = [
  {"instruction": "Qual a temperatura de supply do loop de liquid cooling?",
   "output": "18-22°C, com margem mínima de 3°C acima do dew point local. Fonte: docs/ARCHITECTURE.md"},
  {"instruction": "Qual a eficiência alvo do inverter 800VDC?",
   "output": "Maior que 96% por estágio LLC ressonante, maior que 92% ponta-a-ponta. Fonte: docs/OCP_REFERENCE.md"},
  {"instruction": "O que é Tier V?",
   "output": "Tier V não é certificação oficial do Uptime Institute (que vai até Tier IV). No HelioBus800 é definido como SLA contratual multi-site caso a caso. Fonte: docs/TIER_CERTIFICATION.md"},
  {"instruction": "Qual o CAPEX total estimado até produto certificável?",
   "output": "R$ 20-35 milhões em 3-4 anos, distribuídos em 5 fases. Fonte: docs/CAPEX.md. Dado não garantido — validar com orçamentos reais."},
  {"instruction": "Posso usar diesel no HelioBus800?",
   "output": "Não. O HelioBus800 é 100% solar + BESS, zero diesel. CUE meta = 0. Alerta automático se CUE > 0."}
]
import json
with open('hb800_train.jsonl', 'w') as f:
    for d in training_data:
        f.write(json.dumps(d, ensure_ascii=False) + '\n')
print(f'Dataset: {len(training_data)} exemplos')


In [ ]:
# 3. Fine-tuning leve com Unsloth (Qwen2.5-coder 0.5B, 4-bit, A100 grátis do Colab)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-Coder-0.5B-Instruct',
    max_seq_length=2048, load_in_4bit=True
)
model = FastLanguageModel.get_peft_model(model,
    r=16, target_modules=['q_proj','v_proj'],
    lora_alpha=16, lora_dropout=0, bias='none'
)


In [ ]:
# 4. Treinar e salvar
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
ds = load_dataset('json', data_files='hb800_train.jsonl', split='train')
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
    dataset_text_field='instruction',
    args=TrainingArguments(output_dir='hb800_model', num_train_epochs=3,
        per_device_train_batch_size=2, save_steps=50, logging_steps=10)
)
trainer.train()
model.save_pretrained('hb800_model_final')
tokenizer.save_pretrained('hb800_model_final')
print('Modelo salvo em hb800_model_final/')


In [ ]:
# 5. Zipar e baixar para o Termux
import shutil
shutil.make_archive('hb800_model_final', 'zip', 'hb800_model_final')
from google.colab import files
files.download('hb800_model_final.zip')
print('Baixe o zip e coloque em ~/heliobus800/models/')
print('Para rodar: ollama create hb800-agent -f ~/heliobus800/models/Modelfile')
